# Baseline-Training des Sprachmodells \(M_0\)

Dieses Notebook enthält die für die Bachelorarbeit verwendete Pipeline zum
Vortraining des Basismodells auf *Plain Text Wikipedia -- Simple English*.

Die Konfiguration entspricht dem final verwendeten Baseline-Training
(`clean_baseline_v1`). Das Training war für fünf Epochen bzw. 46.820
Optimizer-Schritte vorgesehen. In der Arbeit wird der bei Schritt 46.460
gespeicherte Zustand als \(M_0\) verwendet.

Nachgelagerte Korpus-, Kandidaten- und Unlearning-Analysen sind bewusst nicht
Teil dieses Notebooks.

## 1. Abhängigkeiten

Die Installationszelle ist nur erforderlich, wenn die Pakete in der aktuellen
Umgebung noch nicht vorhanden sind.

In [ ]:
# Bei Bedarf einmal ausführen:
# %pip install -q tiktoken kagglehub matplotlib

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import random
import shutil
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Verwendetes Gerät:", DEVICE)

## 2. Zentrale Konfiguration

Modell-, Daten- und Trainingsparameter werden an einer Stelle festgelegt.
`RUN_TRAINING=False` verhindert ein versehentliches Training bei „Run all“.

In [ ]:
MODEL_CONFIG = {
    "vocab_size": 50_257,
    "context_length": 1_024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

PATH_CONFIG = {
    "data_dir": Path("/kaggle/input/plain-text-wikipedia-simpleenglish"),
    "cache_dir": Path("cache"),
    "output_dir": Path(
        "/content/drive/MyDrive/model_checkpoints/clean_baseline_v1"
    ),
    # Zum Fortsetzen auf einen vollständigen Trainings-Checkpoint setzen.
    "resume_checkpoint": None,
}

DATA_CONFIG = {
    "train_ratio": 0.90,
    "stride": 1_024,
    "batch_size": 4,
    "num_workers": 0,
}

TRAIN_CONFIG = {
    "n_epochs": 5,
    "peak_lr": 5e-4,
    "initial_lr": 3e-5,
    "min_lr": 1e-6,
    "warmup_steps": 1_000,
    "weight_decay": 0.1,
    "gradient_clip_norm": 1.0,
    "eval_every_steps": 500,
    "eval_batches": 50,
    "sample_every_steps": 2_000,
    "sample_prompt": "A tiger is a",
    "sample_tokens": 50,
    "checkpoint_every_steps": 5_000,
    # Nur für ältere Checkpoints ohne gespeicherten Scheduler-Zustand.
    "legacy_resume_peak_lr": 1e-4,
    "legacy_resume_warmup_steps": 0,
    "seed": 123,
}

RUN_TRAINING = False

set_seed(TRAIN_CONFIG["seed"])
PATH_CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

print(json.dumps(
    {
        "model": MODEL_CONFIG,
        "data": DATA_CONFIG,
        "training": TRAIN_CONFIG,
    },
    indent=2,
))

## 3. Optional: Google Drive einbinden

Diese Zelle ist nur in Google Colab relevant. Sie kann übersprungen werden,
wenn die Checkpoints an einem anderen Pfad gespeichert werden.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Keine Colab-Umgebung erkannt; Google Drive wird nicht eingebunden.")

## 4. Datensatz bereitstellen

Der Datensatz wird nur heruntergeladen, wenn im konfigurierten Verzeichnis
keine Textdatei gefunden wird.

In [ ]:
def find_text_files(data_dir: Path) -> list[Path]:
    return sorted(path for path in data_dir.rglob("*.txt") if path.is_file())


def ensure_simple_english_dataset(data_dir: Path) -> list[Path]:
    existing_files = find_text_files(data_dir)
    if existing_files:
        print(f"Datensatz bereits vorhanden: {len(existing_files)} Textdatei(en)")
        return existing_files

    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "Der Datensatz fehlt und kagglehub ist nicht installiert."
        ) from exc

    print("Lade Simple-English-Wikipedia-Datensatz herunter ...")
    downloaded_path = Path(
        kagglehub.dataset_download("ffatty/plain-text-wikipedia-simpleenglish")
    )

    data_dir.mkdir(parents=True, exist_ok=True)
    if downloaded_path.resolve() != data_dir.resolve():
        shutil.copytree(downloaded_path, data_dir, dirs_exist_ok=True)

    text_files = find_text_files(data_dir)
    if not text_files:
        raise FileNotFoundError(
            f"Nach dem Download wurden in {data_dir} keine .txt-Dateien gefunden."
        )

    print(f"Datensatz bereit: {len(text_files)} Textdatei(en)")
    return text_files


TEXT_FILES = ensure_simple_english_dataset(PATH_CONFIG["data_dir"])
for file_path in TEXT_FILES:
    print(f"- {file_path} ({file_path.stat().st_size / 1e6:.1f} MB)")

## 5. Tokenisierung und robuster Cache

Der Cache-Name hängt von Dateipfaden, Dateigrößen, Änderungszeiten und Tokenizer
ab. Dadurch wird nicht versehentlich ein alter Cache für einen anderen Datensatz
verwendet.

In [ ]:
def torch_load_compatible(
    path: Path,
    *,
    map_location: str | torch.device = "cpu",
    weights_only: bool = False,
) -> Any:
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=weights_only,
        )
    except TypeError:
        # Kompatibilität mit älteren PyTorch-Versionen.
        return torch.load(path, map_location=map_location)


def dataset_fingerprint(file_paths: Iterable[Path], tokenizer_name: str) -> str:
    digest = hashlib.sha256()
    digest.update(tokenizer_name.encode("utf-8"))

    for file_path in sorted(file_paths):
        stat = file_path.stat()
        descriptor = (
            f"{file_path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}"
        )
        digest.update(descriptor.encode("utf-8"))

    return digest.hexdigest()[:16]


def get_pretokenized_data(
    file_paths: list[Path],
    tokenizer,
    cache_dir: Path,
) -> torch.Tensor:
    cache_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = dataset_fingerprint(file_paths, tokenizer.name)
    cache_path = cache_dir / f"simplewiki_gpt2_{fingerprint}.pt"

    if cache_path.exists():
        print(f"Lade tokenisierte Daten aus Cache: {cache_path}")
        token_tensor = torch_load_compatible(
            cache_path,
            map_location="cpu",
            weights_only=True,
        )
        if not isinstance(token_tensor, torch.Tensor) or token_tensor.ndim != 1:
            raise ValueError(f"Ungültiger Token-Cache: {cache_path}")
        return token_tensor.long()

    print("Tokenisiere Datensatz. Dies kann beim ersten Durchlauf dauern.")
    token_parts: list[torch.Tensor] = []

    for index, file_path in enumerate(file_paths, start=1):
        print(f"[{index}/{len(file_paths)}] Tokenisiere {file_path.name}")
        text = file_path.read_text(encoding="utf-8")
        token_ids = tokenizer.encode(
            text,
            allowed_special={"<|endoftext|>"},
        )
        token_parts.append(torch.tensor(token_ids, dtype=torch.long))

        # Dokumentgrenze einfügen, falls mehrere Textdateien vorhanden sind.
        if index < len(file_paths):
            token_parts.append(
                torch.tensor([tokenizer.eot_token], dtype=torch.long)
            )

    token_tensor = (
        token_parts[0]
        if len(token_parts) == 1
        else torch.cat(token_parts, dim=0)
    )
    torch.save(token_tensor, cache_path)

    print(f"Tokenisierung gespeichert: {cache_path}")
    print(f"Tokenanzahl: {len(token_tensor):,}")
    return token_tensor


TOKENIZER = tiktoken.get_encoding("gpt2")
TOKEN_TENSOR = get_pretokenized_data(
    TEXT_FILES,
    TOKENIZER,
    PATH_CONFIG["cache_dir"],
)
print(f"Gesamte Tokenanzahl: {len(TOKEN_TENSOR):,}")

## 6. Speichereffizientes Sliding-Window-Dataset

Das Dataset speichert nicht mehr Hunderttausende einzelne Tensor-Slices in
Python-Listen. Es hält nur den Token-Tensor und berechnet das jeweilige Fenster
erst in `__getitem__`.

In [ ]:
class CachedGPTDataset(Dataset):
    def __init__(
        self,
        token_tensor: torch.Tensor,
        max_length: int,
        stride: int,
    ) -> None:
        if token_tensor.ndim != 1:
            raise ValueError("token_tensor muss eindimensional sein.")
        if max_length <= 0:
            raise ValueError("max_length muss positiv sein.")
        if stride <= 0:
            raise ValueError("stride muss positiv sein.")
        if len(token_tensor) <= max_length:
            raise ValueError(
                "Der Token-Tensor ist zu kurz für ein vollständiges Trainingsfenster."
            )

        self.token_tensor = token_tensor
        self.max_length = max_length
        self.stride = stride
        self.num_windows = 1 + (
            len(token_tensor) - max_length - 1
        ) // stride

    def __len__(self) -> int:
        return self.num_windows

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        if index < 0 or index >= self.num_windows:
            raise IndexError(index)

        start = index * self.stride
        input_ids = self.token_tensor[start : start + self.max_length]
        target_ids = self.token_tensor[start + 1 : start + self.max_length + 1]
        return input_ids, target_ids


def create_dataloaders(
    token_tensor: torch.Tensor,
    *,
    train_ratio: float,
    context_length: int,
    stride: int,
    batch_size: int,
    num_workers: int,
    device: torch.device,
) -> tuple[DataLoader, DataLoader, CachedGPTDataset, CachedGPTDataset]:
    if not 0.0 < train_ratio < 1.0:
        raise ValueError("train_ratio muss zwischen 0 und 1 liegen.")

    split_index = int(train_ratio * len(token_tensor))
    train_tokens = token_tensor[:split_index]
    val_tokens = token_tensor[split_index:]

    train_dataset = CachedGPTDataset(
        train_tokens,
        max_length=context_length,
        stride=stride,
    )
    val_dataset = CachedGPTDataset(
        val_tokens,
        max_length=context_length,
        stride=stride,
    )

    common_loader_args = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": device.type == "cuda",
        "persistent_workers": num_workers > 0,
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        drop_last=True,
        **common_loader_args,
    )
    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        drop_last=False,
        **common_loader_args,
    )

    return train_loader, val_loader, train_dataset, val_dataset


(
    TRAIN_LOADER,
    VAL_LOADER,
    TRAIN_DATASET,
    VAL_DATASET,
) = create_dataloaders(
    TOKEN_TENSOR,
    train_ratio=DATA_CONFIG["train_ratio"],
    context_length=MODEL_CONFIG["context_length"],
    stride=DATA_CONFIG["stride"],
    batch_size=DATA_CONFIG["batch_size"],
    num_workers=DATA_CONFIG["num_workers"],
    device=DEVICE,
)

print(f"Trainingsfenster:   {len(TRAIN_DATASET):,}")
print(f"Validierungsfenster: {len(VAL_DATASET):,}")
print(f"Trainingsschritte pro Epoche: {len(TRAIN_LOADER):,}")
print(
    "Verarbeitete Tokenpositionen pro Epoche: "
    f"{len(TRAIN_LOADER) * DATA_CONFIG['batch_size'] * MODEL_CONFIG['context_length']:,}"
)

### Exakten Train-/Validierungssplit speichern

Die Token-Tensoren werden zusätzlich separat gespeichert, damit spätere
Unlearning- und Referenzmodell-Auswertungen exakt denselben Split verwenden.

In [ ]:
ANALYSIS_CACHE_PATH = Path(
    "/content/drive/MyDrive/"
    "simplewiki_train_tokens_exact.pt"
)

torch.save(
    {
        "train_tokens": (
            TRAIN_DATASET
            .token_tensor
            .detach()
            .cpu()
        ),
        "total_token_count": len(TOKEN_TENSOR),
        "split_index": int(
            DATA_CONFIG["train_ratio"]
            * len(TOKEN_TENSOR)
        ),
        "train_ratio": DATA_CONFIG["train_ratio"],
        "context_length": MODEL_CONFIG["context_length"],
        "stride": DATA_CONFIG["stride"],
    },
    ANALYSIS_CACHE_PATH,
)

print(f"Gespeichert: {ANALYSIS_CACHE_PATH}")

In [ ]:
VAL_TOKENS_PATH = Path(
    "/content/drive/MyDrive/simplewiki_val_tokens_exact.pt"
)

split_index = int(
    DATA_CONFIG["train_ratio"] * len(TOKEN_TENSOR)
)
val_tokens = TOKEN_TENSOR[split_index:].cpu()

torch.save(
    {
        "val_tokens": val_tokens,
        "total_token_count": len(TOKEN_TENSOR),
        "split_index": split_index,
        "train_ratio": DATA_CONFIG["train_ratio"],
        "context_length": MODEL_CONFIG["context_length"],
        "stride": DATA_CONFIG["stride"],
    },
    VAL_TOKENS_PATH,
)

print(f"Validation-Tokens gespeichert: {VAL_TOKENS_PATH}")
print(f"Validation-Tokenanzahl: {len(val_tokens):,}")

## 7. GPT-Modellarchitektur

Die Parameternamen entsprechen der während des Trainings verwendeten
Implementierung und bleiben dadurch mit den gespeicherten Checkpoints kompatibel.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float,
        num_heads: int,
        qkv_bias: bool = False,
    ) -> None:
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError("d_out muss durch num_heads teilbar sein.")

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_tokens, _ = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        queries = queries.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        values = values.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)

        attention_scores = queries @ keys.transpose(2, 3)
        causal_mask = self.mask.bool()[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask, -torch.inf)

        attention_weights = torch.softmax(
            attention_scores / math.sqrt(self.head_dim),
            dim=-1,
        )
        attention_weights = self.dropout(attention_weights)

        context = (attention_weights @ values).transpose(1, 2)
        context = context.reshape(batch_size, num_tokens, self.d_out)
        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim: int) -> None:
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.eps)
        return self.scale * normalized + self.shift


class GELU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return 0.5 * x * (
            1.0
            + torch.tanh(
                math.sqrt(2.0 / math.pi)
                * (x + 0.044715 * x.pow(3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"],
        )
        self.pos_emb = nn.Embedding(
            cfg["context_length"],
            cfg["emb_dim"],
        )
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[
                TransformerBlock(cfg)
                for _ in range(cfg["n_layers"])
            ]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False,
        )

    def forward(self, in_idx: torch.Tensor) -> torch.Tensor:
        _, sequence_length = in_idx.shape
        if sequence_length > self.pos_emb.num_embeddings:
            raise ValueError(
                f"Sequenzlänge {sequence_length} überschreitet "
                f"die Context Length {self.pos_emb.num_embeddings}."
            )

        token_embeddings = self.tok_emb(in_idx)
        position_ids = torch.arange(
            sequence_length,
            device=in_idx.device,
        )
        position_embeddings = self.pos_emb(position_ids)

        x = token_embeddings + position_embeddings
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

## 8. Loss und Evaluation

Trainings- und Validierungsverlust werden während des Trainings in festen
Abständen über die konfigurierte Anzahl von Batches bestimmt.

In [ ]:
def calc_loss_batch(
    input_batch: torch.Tensor,
    target_batch: torch.Tensor,
    model: nn.Module,
    device: torch.device,
) -> torch.Tensor:
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)

    logits = model(input_batch)
    return F.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten(),
    )


@torch.no_grad()
def calc_loss_loader(
    data_loader: DataLoader,
    model: nn.Module,
    device: torch.device,
    num_batches: int | None = None,
) -> float:
    if len(data_loader) == 0:
        return float("nan")

    batches_to_evaluate = (
        len(data_loader)
        if num_batches is None
        else min(num_batches, len(data_loader))
    )

    total_loss = 0.0
    for batch_index, (input_batch, target_batch) in enumerate(data_loader):
        if batch_index >= batches_to_evaluate:
            break
        loss = calc_loss_batch(
            input_batch,
            target_batch,
            model,
            device,
        )
        total_loss += loss.item()

    return total_loss / batches_to_evaluate


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    eval_batches: int,
) -> tuple[float, float]:
    was_training = model.training
    model.eval()

    train_loss = calc_loss_loader(
        train_loader,
        model,
        device,
        num_batches=eval_batches,
    )
    val_loss = calc_loss_loader(
        val_loader,
        model,
        device,
        num_batches=eval_batches,
    )

    model.train(was_training)
    return train_loss, val_loss

## 9. Textgenerierung für Trainingsstichproben

Während des Trainings werden deterministische Stichproben erzeugt, um den
Trainingsfortschritt qualitativ zu kontrollieren.

In [ ]:
@torch.no_grad()
def generate_text(
    model: nn.Module,
    tokenizer,
    prompt: str,
    *,
    max_new_tokens: int,
    context_size: int,
    temperature: float = 0.0,
    top_k: int | None = None,
) -> str:
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens darf nicht negativ sein.")
    if temperature < 0:
        raise ValueError("temperature darf nicht negativ sein.")

    token_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=next(model.parameters()).device,
    ).unsqueeze(0)

    was_training = model.training
    model.eval()

    for _ in range(max_new_tokens):
        conditioned_ids = token_ids[:, -context_size:]
        logits = model(conditioned_ids)[:, -1, :]

        if temperature == 0:
            next_token = torch.argmax(
                logits,
                dim=-1,
                keepdim=True,
            )
        else:
            logits = logits / max(temperature, 1e-8)

            if top_k is not None:
                effective_top_k = min(top_k, logits.shape[-1])
                top_logits, top_indices = torch.topk(
                    logits,
                    effective_top_k,
                    dim=-1,
                )
                probabilities = torch.softmax(top_logits, dim=-1)
                sampled_position = torch.multinomial(
                    probabilities,
                    num_samples=1,
                )
                next_token = torch.gather(
                    top_indices,
                    dim=-1,
                    index=sampled_position,
                )
            else:
                probabilities = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(
                    probabilities,
                    num_samples=1,
                )

        token_ids = torch.cat((token_ids, next_token), dim=1)

    model.train(was_training)
    return tokenizer.decode(token_ids.squeeze(0).tolist())

## 10. Checkpoints und Lernraten-Scheduler

Checkpoints enthalten Modell, Optimizer, Scheduler, Trainingszustand und
Loss-Historie. Dadurch kann ein Lauf konsistent fortgesetzt werden.

In [ ]:
class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        *,
        total_steps: int,
        warmup_steps: int,
        initial_lr: float,
        peak_lr: float,
        min_lr: float,
    ) -> None:
        if total_steps <= 0:
            raise ValueError("total_steps muss positiv sein.")
        if warmup_steps < 0:
            raise ValueError("warmup_steps darf nicht negativ sein.")
        if warmup_steps >= total_steps:
            warmup_steps = max(0, total_steps - 1)
        if not 0 <= min_lr <= peak_lr:
            raise ValueError("Es muss 0 <= min_lr <= peak_lr gelten.")

        self.optimizer = optimizer
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.initial_lr = initial_lr
        self.peak_lr = peak_lr
        self.min_lr = min_lr
        self.step_number = 0

        self._set_lr(self.learning_rate_at(0))

    def learning_rate_at(self, step_number: int) -> float:
        step_number = max(0, step_number)

        if self.warmup_steps > 0 and step_number <= self.warmup_steps:
            fraction = step_number / self.warmup_steps
            return self.initial_lr + fraction * (
                self.peak_lr - self.initial_lr
            )

        denominator = max(1, self.total_steps - self.warmup_steps)
        progress = (
            step_number - self.warmup_steps
        ) / denominator
        progress = min(max(progress, 0.0), 1.0)

        cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + (
            self.peak_lr - self.min_lr
        ) * cosine_factor

    def _set_lr(self, learning_rate: float) -> None:
        for parameter_group in self.optimizer.param_groups:
            parameter_group["lr"] = learning_rate

    def step(self) -> float:
        self.step_number += 1
        learning_rate = self.learning_rate_at(self.step_number)
        self._set_lr(learning_rate)
        return learning_rate

    def state_dict(self) -> dict[str, Any]:
        return {
            "total_steps": self.total_steps,
            "warmup_steps": self.warmup_steps,
            "initial_lr": self.initial_lr,
            "peak_lr": self.peak_lr,
            "min_lr": self.min_lr,
            "step_number": self.step_number,
        }

    def load_state_dict(self, state: dict[str, Any]) -> None:
        self.total_steps = int(state["total_steps"])
        self.warmup_steps = int(state["warmup_steps"])
        self.initial_lr = float(state["initial_lr"])
        self.peak_lr = float(state["peak_lr"])
        self.min_lr = float(state["min_lr"])
        self.step_number = int(state["step_number"])
        self._set_lr(self.learning_rate_at(self.step_number))


def empty_history() -> dict[str, list[float]]:
    return {
        "global_step": [],
        "tokens_seen": [],
        "train_loss": [],
        "val_loss": [],
        "learning_rate": [],
    }


def default_training_state() -> dict[str, int]:
    return {
        "global_step": 0,
        "tokens_seen": 0,
        "completed_epochs": 0,
    }


def move_optimizer_state_to_device(
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> None:
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)


def load_training_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    *,
    device: torch.device,
    fallback_tokens_per_step: int,
) -> tuple[dict[str, int], dict[str, list[float]], dict[str, Any]]:
    checkpoint = torch_load_compatible(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        model.load_state_dict(checkpoint["model_state_dict"])

        if (
            optimizer is not None
            and "optimizer_state_dict" in checkpoint
        ):
            optimizer.load_state_dict(
                checkpoint["optimizer_state_dict"]
            )
            move_optimizer_state_to_device(optimizer, device)

        state = default_training_state()
        state.update(checkpoint.get("training_state", {}))
        state["global_step"] = int(
            checkpoint.get(
                "global_step",
                state.get("global_step", 0),
            )
        )

        if state.get("tokens_seen", 0) == 0 and state["global_step"] > 0:
            state["tokens_seen"] = (
                state["global_step"] * fallback_tokens_per_step
            )
            print(
                "Hinweis: Der alte Checkpoint enthält keine Tokenzahl. "
                "Sie wurde aus global_step angenähert."
            )

        history = checkpoint.get("history", empty_history())
        return state, history, checkpoint

    if not isinstance(checkpoint, dict):
        raise TypeError("Der Checkpoint besitzt ein unbekanntes Format.")

    # Rückwärtskompatibilität mit einem reinen model.state_dict().
    model.load_state_dict(checkpoint)
    return default_training_state(), empty_history(), {}


def save_training_checkpoint(
    checkpoint_path: Path,
    *,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: WarmupCosineScheduler,
    training_state: dict[str, int],
    history: dict[str, list[float]],
    model_config: dict[str, Any],
) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "format_version": 2,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "training_state": dict(training_state),
        "global_step": int(training_state["global_step"]),
        "history": history,
        "model_config": model_config,
    }
    torch.save(checkpoint, checkpoint_path)


def create_scheduler(
    optimizer: torch.optim.Optimizer,
    *,
    steps_to_run: int,
    checkpoint: dict[str, Any],
    resumed: bool,
) -> WarmupCosineScheduler:
    saved_scheduler_state = checkpoint.get("scheduler_state_dict")

    if saved_scheduler_state is not None:
        scheduler = WarmupCosineScheduler(
            optimizer,
            total_steps=max(1, int(saved_scheduler_state["total_steps"])),
            warmup_steps=int(saved_scheduler_state["warmup_steps"]),
            initial_lr=float(saved_scheduler_state["initial_lr"]),
            peak_lr=float(saved_scheduler_state["peak_lr"]),
            min_lr=float(saved_scheduler_state["min_lr"]),
        )
        scheduler.load_state_dict(saved_scheduler_state)

        remaining_steps = max(
            0,
            scheduler.total_steps - scheduler.step_number,
        )
        print(
            "Scheduler-Zustand wiederhergestellt. "
            f"Verbleibende geplante Schritte: {remaining_steps:,}"
        )
        if steps_to_run > remaining_steps:
            print(
                "Hinweis: Die angeforderten Schritte überschreiten den "
                "gespeicherten Scheduler-Plan. Nach dessen Ende bleibt "
                "die Lernrate bei min_lr."
            )
        return scheduler

    if resumed:
        print(
            "Alter Checkpoint ohne Scheduler-Zustand: "
            "Es beginnt eine neue LR-Phase."
        )
        peak_lr = TRAIN_CONFIG["legacy_resume_peak_lr"]
        warmup_steps = TRAIN_CONFIG["legacy_resume_warmup_steps"]
        initial_lr = peak_lr if warmup_steps == 0 else min(
            TRAIN_CONFIG["initial_lr"],
            peak_lr,
        )
    else:
        peak_lr = TRAIN_CONFIG["peak_lr"]
        warmup_steps = TRAIN_CONFIG["warmup_steps"]
        initial_lr = TRAIN_CONFIG["initial_lr"]

    return WarmupCosineScheduler(
        optimizer,
        total_steps=max(1, steps_to_run),
        warmup_steps=warmup_steps,
        initial_lr=initial_lr,
        peak_lr=peak_lr,
        min_lr=TRAIN_CONFIG["min_lr"],
    )

## 11. Trainingsfunktion

Die Trainingsfunktion kombiniert Lernraten-Scheduler, Gradient Clipping,
Monitoring, Stichprobengenerierung und Checkpointing.

In [ ]:
def print_training_sample(
    model: nn.Module,
    tokenizer,
    prompt: str,
    *,
    max_new_tokens: int,
) -> None:
    sample = generate_text(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
        context_size=model.pos_emb.num_embeddings,
        temperature=0.0,
    )
    print("\n--- Deterministische Stichprobe ---")
    print(sample)
    print("------------------------------------\n")


def train_model(
    *,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: WarmupCosineScheduler,
    device: torch.device,
    tokenizer,
    n_epochs: int,
    output_dir: Path,
    training_state: dict[str, int],
    history: dict[str, list[float]],
) -> tuple[dict[str, list[float]], dict[str, int]]:
    if n_epochs <= 0:
        raise ValueError("n_epochs muss positiv sein.")

    try:
        for local_epoch in range(1, n_epochs + 1):
            displayed_epoch = training_state["completed_epochs"] + 1
            print(
                f"Starte Notebook-Epoche {displayed_epoch} "
                f"({local_epoch}/{n_epochs} in dieser Ausführung) ..."
            )

            model.train()

            for input_batch, target_batch in train_loader:
                learning_rate = scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                loss = calc_loss_batch(
                    input_batch,
                    target_batch,
                    model,
                    device,
                )
                loss.backward()

                gradient_clip = TRAIN_CONFIG["gradient_clip_norm"]
                if gradient_clip is not None:
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=gradient_clip,
                    )

                optimizer.step()

                training_state["global_step"] += 1
                training_state["tokens_seen"] += input_batch.numel()
                global_step = training_state["global_step"]

                if global_step % TRAIN_CONFIG["eval_every_steps"] == 0:
                    train_loss, val_loss = evaluate_model(
                        model,
                        train_loader,
                        val_loader,
                        device,
                        TRAIN_CONFIG["eval_batches"],
                    )

                    history["global_step"].append(global_step)
                    history["tokens_seen"].append(
                        training_state["tokens_seen"]
                    )
                    history["train_loss"].append(train_loss)
                    history["val_loss"].append(val_loss)
                    history["learning_rate"].append(learning_rate)

                    print(
                        f"Ep {displayed_epoch} "
                        f"(Step {global_step:07d}): "
                        f"Train {train_loss:.3f}, "
                        f"Val {val_loss:.3f}, "
                        f"LR {learning_rate:.2e}"
                    )

                if (
                    TRAIN_CONFIG["sample_every_steps"] > 0
                    and global_step
                    % TRAIN_CONFIG["sample_every_steps"]
                    == 0
                ):
                    print_training_sample(
                        model,
                        tokenizer,
                        TRAIN_CONFIG["sample_prompt"],
                        max_new_tokens=TRAIN_CONFIG["sample_tokens"],
                    )

                if (
                    TRAIN_CONFIG["checkpoint_every_steps"] > 0
                    and global_step
                    % TRAIN_CONFIG["checkpoint_every_steps"]
                    == 0
                ):
                    checkpoint_path = (
                        output_dir
                        / f"model_step_{global_step:07d}.pth"
                    )
                    save_training_checkpoint(
                        checkpoint_path,
                        model=model,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        training_state=training_state,
                        history=history,
                        model_config=MODEL_CONFIG,
                    )
                    print(f"Checkpoint gespeichert: {checkpoint_path}")

            training_state["completed_epochs"] += 1

            print_training_sample(
                model,
                tokenizer,
                TRAIN_CONFIG["sample_prompt"],
                max_new_tokens=TRAIN_CONFIG["sample_tokens"],
            )

            epoch_checkpoint = (
                output_dir
                / (
                    f"model_epoch_"
                    f"{training_state['completed_epochs']:03d}_"
                    f"step_{training_state['global_step']:07d}.pth"
                )
            )
            save_training_checkpoint(
                epoch_checkpoint,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                training_state=training_state,
                history=history,
                model_config=MODEL_CONFIG,
            )
            print(f"Epochen-Checkpoint gespeichert: {epoch_checkpoint}")

    except KeyboardInterrupt:
        interrupted_path = (
            output_dir
            / (
                f"model_step_"
                f"{training_state['global_step']:07d}_interrupted.pth"
            )
        )
        save_training_checkpoint(
            interrupted_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            training_state=training_state,
            history=history,
            model_config=MODEL_CONFIG,
        )
        print(f"Unterbrechungs-Checkpoint gespeichert: {interrupted_path}")

    return history, training_state

## 12. Modell und Optimizer initialisieren oder Checkpoint laden

Ohne `resume_checkpoint` wird ein neues Modell initialisiert. Ein vollständiger
Checkpoint stellt zusätzlich Optimizer-, Scheduler- und Trainingszustand wieder her.

In [ ]:
MODEL = GPTModel(MODEL_CONFIG).to(DEVICE)
OPTIMIZER = torch.optim.AdamW(
    MODEL.parameters(),
    lr=TRAIN_CONFIG["peak_lr"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)

TRAINING_STATE = default_training_state()
HISTORY = empty_history()
LOADED_CHECKPOINT: dict[str, Any] = {}
RESUMED = False

resume_path = PATH_CONFIG["resume_checkpoint"]
if resume_path is not None and resume_path.exists():
    print(f"Lade Checkpoint: {resume_path}")
    (
        TRAINING_STATE,
        HISTORY,
        LOADED_CHECKPOINT,
    ) = load_training_checkpoint(
        resume_path,
        MODEL,
        OPTIMIZER,
        device=DEVICE,
        fallback_tokens_per_step=(
            DATA_CONFIG["batch_size"]
            * MODEL_CONFIG["context_length"]
        ),
    )
    RESUMED = True
    print(
        f"Checkpoint geladen. Global Step: "
        f"{TRAINING_STATE['global_step']:,}"
    )
else:
    if resume_path is not None:
        print(
            f"Checkpoint nicht gefunden: {resume_path}\n"
            "Es wird ein neues Modell verwendet."
        )
    else:
        print("Neues Modell wird von Grund auf trainiert.")

STEPS_TO_RUN = len(TRAIN_LOADER) * TRAIN_CONFIG["n_epochs"]
SCHEDULER = create_scheduler(
    OPTIMIZER,
    steps_to_run=STEPS_TO_RUN,
    checkpoint=LOADED_CHECKPOINT,
    resumed=RESUMED,
)

parameter_count = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)
print(f"Modellparameter: {parameter_count:,}")
print(f"Geplante zusätzliche Schritte: {STEPS_TO_RUN:,}")
print(
    "Aktuelle Lernrate:",
    OPTIMIZER.param_groups[0]["lr"],
)

## 13. Training ausführen

Setze in der Konfigurationszelle `RUN_TRAINING = True`, um diese Zelle aktiv
trainieren zu lassen. Dadurch kann das gesamte Notebook gefahrlos mit
„Run all“ initialisiert werden.

In [ ]:
if RUN_TRAINING:
    HISTORY, TRAINING_STATE = train_model(
        model=MODEL,
        train_loader=TRAIN_LOADER,
        val_loader=VAL_LOADER,
        optimizer=OPTIMIZER,
        scheduler=SCHEDULER,
        device=DEVICE,
        tokenizer=TOKENIZER,
        n_epochs=TRAIN_CONFIG["n_epochs"],
        output_dir=PATH_CONFIG["output_dir"],
        training_state=TRAINING_STATE,
        history=HISTORY,
    )

    final_checkpoint = (
        PATH_CONFIG["output_dir"]
        / f"model_final_step_{TRAINING_STATE['global_step']:07d}.pth"
    )
    save_training_checkpoint(
        final_checkpoint,
        model=MODEL,
        optimizer=OPTIMIZER,
        scheduler=SCHEDULER,
        training_state=TRAINING_STATE,
        history=HISTORY,
        model_config=MODEL_CONFIG,
    )
    print(f"Finaler Checkpoint gespeichert: {final_checkpoint}")
else:
    print(
        "Training ist deaktiviert. "
        "Setze RUN_TRAINING = True und führe die Konfigurations-, "
        "Initialisierungs- und Trainingszelle erneut aus."
    )

## 14. Loss-Verlauf darstellen

Die x-Achse verwendet globale Optimizer-Schritte, sodass die Darstellung auch
nach einem Resume eindeutig bleibt.

In [ ]:
def plot_training_history(
    history: dict[str, list[float]],
    *,
    save_path: Path | None = None,
) -> None:
    if not history["global_step"]:
        print("Noch keine Evaluationswerte in der Historie vorhanden.")
        return

    figure, axis = plt.subplots(figsize=(8, 4.5))
    axis.plot(
        history["global_step"],
        history["train_loss"],
        label="Training Loss",
    )
    axis.plot(
        history["global_step"],
        history["val_loss"],
        linestyle="-.",
        label="Validation Loss",
    )
    axis.set_xlabel("Globaler Optimizer-Schritt")
    axis.set_ylabel("Cross-Entropy-Loss")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_path, bbox_inches="tight")
        print(f"Loss-Plot gespeichert: {save_path}")

    plt.show()


plot_training_history(
    HISTORY,
    save_path=PATH_CONFIG["output_dir"] / "loss_plot.pdf",
)

## Verwendeter Baseline-Zustand

Für die weiteren Experimente der Arbeit wurde
`model_final_step_0046460.pth` als Basismodell \(M_0\) verwendet.